### ASCII Operators

The following Python functions are from the course notes:

In [ ]:
def runwasm(wasmfile):
    from IPython.display import display, Javascript
    with open(wasmfile, "rb") as file: 
        wasm_bytes = file.read() # read the WASM file and convert its content to a comma-separated string of byte values
        wasm_byte_array_str = ','.join(str(byte) for byte in wasm_bytes)
 
    display(Javascript(f""" // Javascript code to compile and run the WASM module
    const wasmBinary = new Uint8Array([{wasm_byte_array_str}]); // convert back to binary representation
 
    const params = {{
        'P0lib': {{
            'write': i => element.append(i + ' '),
            'writeln': () => element.append(document.createElement('br')),
            'read': () => window.prompt()
        }}
    }};
 
    WebAssembly.compile(wasmBinary.buffer) // compile shareable code
        .then(module => WebAssembly.instantiate(module, params)) // create an instance with memory
        // .then(instance => instance.exports.program()); // run the main program; not needed if a start function is specified
    """))

In [ ]:
def runpywasm(wasmfile):
    from pywasm.core import Machine, Runtime, FuncType, ValType
    # P0lib implementation in Python
    def write(_: Machine, args: list[int]) -> list[int]:
        print(args[0]); return []
    def writeln(_: Machine, args: list[int]) -> list[int]:
        print(); return []
    def read(_: Machine, args: list[int]) -> list[int]:
        return [int(input())]
    # Create runtime
    runtime = Runtime()
    runtime.imports = {'P0lib':
        {'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
         'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
         'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read)}}
    # Create and run instance
    instance = runtime.instance_from_file(wasmfile)

In [ ]:
def runwasmtime(wasmfile):
    from wasmtime import Engine, Store, Module, Linker, Func, FuncType, ValType
    # P0lib implementation in Python
    def write(i: int): print(i)
    def writeln(): print()
    def read() -> int: return int(input())
    # Create engine, store, and module
    engine = Engine()
    store = Store(engine)
    module = Module(store.engine, open(wasmfile, 'rb').read())
    # Use Linker to create the P0lib library
    write_func = Func(store, FuncType([ValType.i32()], []), write)
    writeln_func = Func(store, FuncType([], []), writeln)
    read_func = Func(store, FuncType([], [ValType.i32()]), read)
    linker = Linker(engine)
    linker.define(store, "P0lib", "write", write_func)
    linker.define(store, "P0lib", "writeln", writeln_func)
    linker.define(store, "P0lib", "read", read_func)
    # Create and run an instance
    instance = linker.instantiate(store, module)

In [ ]:
import import_ipynb
from P0 import compileString

Consider the following example from the course notes:

In [ ]:
compileString("""
procedure quotrem(x, y: integer) → (q, r: integer)
    q, r := 0, x
    while r ≥ y do // q × y + r = x ∧ r ≥ y
        r, q := r - y, q + 1

program arithmetic
    var x, y, q, r: integer
      x ← read(); y ← read()
      q, r ← quotrem(x, y)
      write(q); write(r)
""", "arith.wat")

In [ ]:
!wat2wasm arith.wat

Run the following cells:

In [ ]:
runwasm("arith.wasm")

In [ ]:
runpywasm("arith.wasm")

In [ ]:
runwasmtime("arith.wasm")

Suppose you like to have ASCII alternatives to Unicode characters:

- `*` for `×`
- `!=`, `<=`, `>=` for `≠`, `≤`, `≥`

Modify the P0 compiler that comes with this lab such that both Unicode characters and ASCII equivalents are allowed. Which parts of the compiler and which files need to be modified?

YOUR ANSWER HERE

Here is a test:

In [ ]:
compileString("""
program arithmetic
    var x, y: integer
      x ← read(); y ← read()
      if x != y then write(x * y)
      if (x <= y) or (x >= y) then write(x + y)
""", "ascii.wat")

In [ ]:
!wat2wasm ascii.wat

In [ ]:
runwasm("ascii.wasm")

In [ ]:
runpywasm("ascii.wasm")

In [ ]:
runwasmtime("ascii.wasm")